In [2]:
import numpy as np
import scipy.optimize as opt
from scipy.special import erfc, erfcinv

from optic.models.devices import mzm, photodiode
from optic.models.channels import linearFiberChannel
from optic.comm.modulation import modulateGray
from optic.comm.sources import bitSource
from optic.dsp.core import upsample, pulseShape, anorm, firFilter
from optic.utils import parameters, dBm2W
from optic.comm.metrics import bert

In [18]:
# simulation parameters
SpS = 16  # samples per symbol
M = 2  # order of the modulation format
Rs = 10e9  # Symbol rate
Fs = SpS * Rs  # Signal sampling frequency (samples/second)
Pi_dBm = 3  # laser optical power at the input of the MZM in dBm
Pi = dBm2W(Pi_dBm)  # convert from dBm to W

# Bit source parameters
paramBits = parameters()
paramBits.nBits = 2**18  # number of bits to be generated
paramBits.mode = 'random'
paramBits.seed = 123

# pulse shaping parameters
paramPulse = parameters()
paramPulse.pulseType = 'nrz'
paramPulse.SpS = SpS

# MZM parameters
paramMZM = parameters()
paramMZM.Vpi = 2
paramMZM.Vb = -paramMZM.Vpi / 2

# linear fiber optical channel parameters
paramCh = parameters()
paramCh.L = 100
paramCh.alpha = 0.2
paramCh.D = 16
paramCh.Fc = 193.1e12
paramCh.Fs = Fs

# photodiode parameters
paramPD = parameters()
paramPD.ideal = False
paramPD.B = Rs
paramPD.Fs = Fs
paramPD.seed = 456

# Optical transmission chain
def optical_chain(symbTx):
    # upsampling
    symbolsUp = upsample(symbTx, SpS)

    # pulse shaping
    pulse = pulseShape(paramPulse)
    sigTx = firFilter(pulse, symbolsUp)
    sigTx = anorm(sigTx)  # normalize to 1 Vpp

    # artificial distortion (non linear)
    alpha = 0.5
    sigTx = sigTx + alpha * sigTx**3

    # optical modulation
    Ai = np.sqrt(Pi)
    sigTxo = mzm(Ai, sigTx, paramMZM)

    # linear fiber channel model
    sigCh = linearFiberChannel(sigTxo, paramCh)

    # noisy PD
    I_Rx = photodiode(sigCh, paramPD)

    # symbol-rate samples
    I_Rx = I_Rx[0::SpS]

    return I_Rx

In [19]:
#Utility Functions
# Helper functions for normalization and metrics
def crop_same_length(a, b):
    """Ensure two signals have the same length"""
    n = min(len(a), len(b))
    return np.asarray(a[:n]), np.asarray(b[:n])


def normalize_signal(x, eps=1e-12):
    """Normalize signal to zero mean and unit variance"""
    mu = np.mean(x)
    std = np.std(x) + eps
    return (x-mu)/std, mu, std


def apply_norm(x, mu, std):
    """Apply previously computed normalization"""
    return (x-mu)/(std+1e-12)


def nmse_db(y_true, y_pred):
    """Compute normalized MSE in dB"""
    y_true, y_pred = crop_same_length(y_true, y_pred)
    err = np.sum((y_true-y_pred)**2)
    sig = np.sum(y_true**2) + 1e-12
    return 10*np.log10(err/sig)


def calc_evm_percent(rx, ref):
    """Compute EVM percentage"""
    rx, ref = crop_same_length(rx, ref)

    rx = (rx-np.mean(rx))/(np.std(rx)+1e-12)
    ref = (ref-np.mean(ref))/(np.std(ref)+1e-12)

    evm = np.sqrt(np.mean((rx-ref)**2)/(np.mean(ref**2)+1e-12))
    return 100*evm

In [20]:
# Wiener-Hammerstein DPD Model
# Structure:
# z[n] = L2( N( L1(x[n]) ) 

class WHDPD:

    def __init__(self, L1=7, L2=7):

        # FIR lengths
        self.L1 = L1
        self.L2 = L2

        # Initialize FIR filters
        self.g1 = np.zeros(L1)
        self.g1[L1//2] = 1

        self.g2 = np.zeros(L2)
        self.g2[L2//2] = 1

        # Polynomial coefficients
        self.a = np.array([1.0,0.0,0.0])   # [a1,a3,a5]


    def fir(self, x, h):
        """FIR convolution"""
        return np.convolve(x, h, mode='same')


    def nonlinear(self, u):
        """Odd-order memoryless polynomial"""
        return self.a[0]*u + self.a[1]*u*(np.abs(u)**2) + self.a[2]*u*(np.abs(u)**4)


    def forward(self, x):
        """WH forward model"""

        u = self.fir(x, self.g1)     # L1
        w = self.nonlinear(u)        # N
        z = self.fir(w, self.g2)     # L2

        return z


    def get_params(self):
        """Return parameter vector"""
        return np.concatenate([self.g1,self.a,self.g2])


    def set_params(self,p):
        """Update model parameters"""
        self.g1 = p[:self.L1]
        self.a = p[self.L1:self.L1+3]
        self.g2 = p[self.L1+3:self.L1+3+self.L2]

In [7]:
# Optimization loss
def wh_loss(p, model, x, y):
    model.set_params(p)
    y_hat = model.forward(x)
    err = y_hat-y
    return np.mean(err**2)

In [22]:
# Training data generation (ILA)
bits_train = bitSource(paramBits)
symb_train = modulateGray(bits_train, M, "pam")

rx_train = optical_chain(symb_train)

symb_train,rx_train = crop_same_length(symb_train,rx_train)

rx_train_n,rx_mu,rx_std = normalize_signal(rx_train)
tx_train_n,tx_mu,tx_std = normalize_signal(symb_train)

In [23]:
# ILA training loop
L1,L2 = 7,7

model = WHDPD(L1,L2)

p = model.get_params()

n_iter = 5

for i in range(n_iter):

    res = opt.minimize(
        wh_loss,
        p,
        args=(model,rx_train_n,tx_train_n),
        method='L-BFGS-B',
        options={'maxiter':80}
    )

    p = res.x
    model.set_params(p)

    print(f"ILA iteration {i+1} completed")

ILA iteration 1 completed
ILA iteration 2 completed
ILA iteration 3 completed
ILA iteration 4 completed
ILA iteration 5 completed


In [24]:
# Predistorter is the trained postdistorter

dpd = WHDPD(L1,L2)
dpd.set_params(model.get_params())

coeffs = len(dpd.get_params())

train_nmse = nmse_db(tx_train_n, model.forward(rx_train_n))

In [25]:
def optical_chain_dpd(symb,DPD_ON=False):

    if DPD_ON:
        x = apply_norm(symb,tx_mu,tx_std)
        z = dpd.forward(x)
    else:
        z = symb.copy()
    return optical_chain(z)

In [26]:
print("\nStarting DPD simulation...",end=" ")

paramBits.seed = 444

bits_test = bitSource(paramBits)
symb_test = modulateGray(bits_test,M,"pam")

rx_before = optical_chain_dpd(symb_test,False)
rx_after  = optical_chain_dpd(symb_test,True)

print("done.")


Starting DPD simulation... done.


In [29]:
BER_before,Q_before = bert(rx_before,bits_test)
BER_after,Q_after = bert(rx_after,bits_test)

EVM_before = calc_evm_percent(rx_before,symb_test)
EVM_after = calc_evm_percent(rx_after,symb_test)

SNR_before = -20*np.log10(EVM_before/100)
SNR_after = -20*np.log10(EVM_after/100)

NMSE_before = nmse_db(symb_test,rx_before)
NMSE_after = nmse_db(symb_test,rx_after)

print("\n>>> Final Performance Metrics After WH-DPD <<<")

print(f"Final SER = {BER_after:.3e}")
print(f"Final BER = {BER_after:.3e}")
print(f"Final SNR = {SNR_after:.3f} dB")
print(f"Final EVM = {EVM_after:.3f} %")
print(f"Final Q-factor = {Q_after:.3f}")
print(f"Final Pb = {BER_after:.3e}")

print("\n>>> Comparison <<<")

print(
f"Before DPD: BER = {BER_before:.3e}, Qf = {Q_before:.3f}, "
f"SNR = {SNR_before:.3f} dB, EVM = {EVM_before:.3f} % "
)

print(
f"After DPD: BER = {BER_after:.3e}, Qf = {Q_after:.3f}, "
f"SNR = {SNR_after:.3f} dB, EVM = {EVM_after:.3f} % "
)

print("\nTraining")
print(f"Coefficients = {coeffs}")
print(f"Train NMSE(dB) = {train_nmse:.2f}")


>>> Final Performance Metrics After WH-DPD <<<
Final SER = 1.366e-03
Final BER = 1.366e-03
Final SNR = 9.583 dB
Final EVM = 33.180 %
Final Q-factor = 2.908
Final Pb = 1.366e-03

>>> Comparison <<<
Before DPD: BER = 3.288e-03, Qf = 2.581, SNR = 8.564 dB, EVM = 37.308 % 
After DPD: BER = 1.366e-03, Qf = 2.908, SNR = 9.583 dB, EVM = 33.180 % 

Training
Coefficients = 17
Train NMSE(dB) = -15.06
